# Olist 巴西電商資料分析

## 分析目標
以巴西電商平台 Olist 2016–2018 訂單資料，回答：
1. **業績**：營收成長趨勢與主要貢獻品類為何？
2. **體驗**：客戶評分分布與物流效率如何？哪些州表現好/差？
3. **客戶**：以 RFM 分群，找出高價值與高流失風險客戶。

## 工具
SQLite + SQL + pandas + matplotlib + Tableau

## 資料來源
[Olist Brazilian E-Commerce Dataset (Kaggle)](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

## 1. 載入套件與資料庫設定

In [ ]:
import os
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Microsoft JhengHei'
matplotlib.rcParams['axes.unicode_minus'] = False

# 路徑設定（相對路徑，方便跨環境執行）
DATA_DIR = os.path.join('..', 'data')
DB_PATH = os.path.join('..', 'olist.db')
OUTPUT_DIR = os.path.join('..', 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('套件載入成功')

## 2. 建立 SQLite 資料庫

將 Kaggle 下載的 9 張 CSV 一次匯入 SQLite，後續分析全用 SQL 完成。

In [ ]:
conn = sqlite3.connect(DB_PATH)

tables = {
    'orders':                'olist_orders_dataset.csv',
    'order_items':           'olist_order_items_dataset.csv',
    'order_payments':        'olist_order_payments_dataset.csv',
    'order_reviews':         'olist_order_reviews_dataset.csv',
    'customers':             'olist_customers_dataset.csv',
    'sellers':               'olist_sellers_dataset.csv',
    'products':              'olist_products_dataset.csv',
    'category_translation':  'product_category_name_translation.csv',
    'geolocation':           'olist_geolocation_dataset.csv',
}

for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f'{table_name:22s} 載入 {len(df):>8,} 筆')

print('\n所有資料載入完成')

## 3. 核心 KPI 與四大主視覺

四個主要分析：
- 月營收趨勢（2017–2018）
- 熱銷商品類別 Top 10
- 客戶評分分布
- 各州總營收 Top 10

In [ ]:
df_revenue = pd.read_sql('''
    SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS 月份,
           ROUND(SUM(oi.price), 2) AS 總營收
    FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
      AND strftime('%Y-%m', o.order_purchase_timestamp) >= '2017-01'
    GROUP BY 月份 ORDER BY 月份
''', conn)

df_category = pd.read_sql('''
    SELECT ct.product_category_name_english AS 商品類別,
           ROUND(SUM(oi.price), 2) AS 總營收
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN category_translation ct ON p.product_category_name = ct.product_category_name
    JOIN orders o ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY 商品類別 ORDER BY 總營收 DESC LIMIT 10
''', conn)

df_reviews = pd.read_sql('''
    SELECT review_score AS 評分, COUNT(*) AS 數量
    FROM order_reviews GROUP BY review_score ORDER BY review_score
''', conn)

df_state = pd.read_sql('''
    SELECT c.customer_state AS 州, ROUND(SUM(oi.price), 2) AS 總營收
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY 州 ORDER BY 總營收 DESC LIMIT 10
''', conn)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes[0,0].plot(df_revenue['月份'], df_revenue['總營收'], marker='o')
axes[0,0].set_title('月營收趨勢'); axes[0,0].tick_params(axis='x', rotation=45)
axes[0,1].barh(df_category['商品類別'], df_category['總營收']); axes[0,1].set_title('熱銷商品類別 Top 10')
axes[1,0].bar(df_reviews['評分'], df_reviews['數量']); axes[1,0].set_title('客戶評分分布')
axes[1,1].bar(df_state['州'], df_state['總營收']); axes[1,1].set_title('各州總營收 Top 10')
plt.tight_layout(); plt.show()

**Insight**
- 月營收從 2017-01 的 ~140K 一路成長至 2017-11 的 ~1M（黑五 + 雙 11 帶動高峰）
- 2018 年趨於穩定（0.7–1.0M），顯示平台進入成熟期
- 健康美妝、watches_gifts、bed_bath_table 為三大營收支柱
- 評分集中在 5 星（佔 ~58%），但 1 星仍有 ~12%，需關注負評來源

**Recommendation**
- 行銷預算應在 Q4 前置，搭配 Top 3 品類做主推
- 1 星訂單應與物流延遲、商品描述不符做交叉分析（見下節）

## 4. 物流效率：預期 vs 實際送達天數

In [ ]:
df_logistics = pd.read_sql('''
    SELECT
        c.customer_state AS 州,
        ROUND(AVG(JULIANDAY(o.order_delivered_customer_date) -
                  JULIANDAY(o.order_purchase_timestamp)), 1) AS 實際天數,
        ROUND(AVG(JULIANDAY(o.order_estimated_delivery_date) -
                  JULIANDAY(o.order_purchase_timestamp)), 1) AS 預期天數
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY 州
    ORDER BY 實際天數 ASC
    LIMIT 15
''', conn)

fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(df_logistics))
ax.bar(x, df_logistics['預期天數'], width=0.4, label='預期天數', color='#90CAF9')
ax.bar([i + 0.4 for i in x], df_logistics['實際天數'], width=0.4, label='實際天數', color='#1565C0')
ax.set_title('各州物流效率：預期 vs 實際送達天數', fontsize=14, fontweight='bold')
ax.set_xticks([i + 0.2 for i in x]); ax.set_xticklabels(df_logistics['州'])
ax.set_ylabel('天數'); ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'logistics.png'), dpi=150, bbox_inches='tight')
plt.show()

**Insight**
- 全國平均實際送達 15.4 天，比預期 ~24 天提早 ~36%（平台普遍給保守 ETA）
- SP（聖保羅）僅 8.8 天，物流最強；RN/BA 仍需 19+ 天，差距明顯
- 預期 vs 實際的 gap 在偏遠州（MT/PE/BA/RN）特別大，代表 ETA 給太寬鬆，使用者體感雖好但可能影響購買決策

**Recommendation**
- 將 SP 倉儲/配送模式逐步複製到 RJ、MG、RS 等高訂單州，可進一步壓低送達天數
- 偏遠州 ETA 過於保守 → 行銷頁可改顯示「預估 X 天送達」更積極的數字以提升轉換

## 5. 匯出 CSV 給 Tableau

In [ ]:
for name, df in [
    ('revenue', df_revenue),
    ('category', df_category),
    ('reviews', df_reviews),
    ('state', df_state),
    ('logistics', df_logistics),
]:
    df.to_csv(os.path.join(OUTPUT_DIR, f'{name}.csv'), index=False)
print('CSV 匯出完成')

## 6. 整體 KPI 與年度成長

In [ ]:
kpi = pd.read_sql('''
    SELECT
        ROUND(SUM(oi.price)/1000000, 2) AS 總營收_M,
        COUNT(DISTINCT o.order_id)      AS 總訂單數,
        ROUND(AVG(r.review_score), 1)   AS 平均評分,
        ROUND(AVG(JULIANDAY(o.order_delivered_customer_date) -
                  JULIANDAY(o.order_purchase_timestamp)), 1) AS 平均送達天數
    FROM orders o
    JOIN order_items oi   ON o.order_id = oi.order_id
    JOIN order_reviews r  ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
''', conn)
kpi

In [ ]:
kpi_yearly = pd.read_sql('''
    SELECT
        strftime('%Y', o.order_purchase_timestamp) AS 年份,
        ROUND(SUM(oi.price)/1000000, 2) AS 總營收_M,
        COUNT(DISTINCT o.order_id)      AS 總訂單數,
        ROUND(AVG(r.review_score), 1)   AS 平均評分,
        ROUND(AVG(JULIANDAY(o.order_delivered_customer_date) -
                  JULIANDAY(o.order_purchase_timestamp)), 1) AS 平均送達天數
    FROM orders o
    JOIN order_items oi   ON o.order_id = oi.order_id
    JOIN order_reviews r  ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY 年份
''', conn)
kpi_yearly.to_csv(os.path.join(OUTPUT_DIR, 'kpi_yearly.csv'), index=False)
kpi_yearly

## 7. RFM 客戶分群分析

**為什麼做 RFM？**
營收與訂單只看到「公司賺多少」，看不到「客戶結構」。RFM 是電商分群最常用的方法：
- **R (Recency)**：距最後一次購買天數
- **F (Frequency)**：購買次數
- **M (Monetary)**：累積消費金額

以最後一筆訂單日期為觀察點，用 NTILE(5) 對每個維度打 1–5 分，再以分數組合分群。

In [ ]:
rfm = pd.read_sql('''
    WITH snapshot AS (
        SELECT MAX(order_purchase_timestamp) AS snapshot_date
        FROM orders WHERE order_status = 'delivered'
    ),
    rfm_base AS (
        SELECT
            c.customer_unique_id,
            ROUND(JULIANDAY((SELECT snapshot_date FROM snapshot)) -
                  JULIANDAY(MAX(o.order_purchase_timestamp)), 0) AS recency_days,
            COUNT(DISTINCT o.order_id) AS frequency,
            ROUND(SUM(oi.price), 2)    AS monetary
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        JOIN customers c    ON o.customer_id = c.customer_id
        WHERE o.order_status = 'delivered'
        GROUP BY c.customer_unique_id
    )
    SELECT
        customer_unique_id,
        recency_days,
        frequency,
        monetary,
        NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
        NTILE(5) OVER (ORDER BY frequency ASC)     AS f_score,
        NTILE(5) OVER (ORDER BY monetary ASC)      AS m_score
    FROM rfm_base
''', conn)

def segment(row):
    r, f, m = row['r_score'], row['f_score'], row['m_score']
    if r >= 4 and f >= 4 and m >= 4: return '冠軍客戶'
    if r >= 4 and f >= 3:            return '忠誠客戶'
    if r >= 4:                       return '潛力新客'
    if r <= 2 and f >= 3:            return '流失風險'
    if r <= 2:                       return '已流失'
    return '一般客戶'

rfm['segment'] = rfm.apply(segment, axis=1)
rfm.head()

In [ ]:
segment_summary = (
    rfm.groupby('segment')
       .agg(客戶數=('customer_unique_id', 'count'),
            平均_R_天=('recency_days', 'mean'),
            平均_F_次=('frequency', 'mean'),
            平均_M_元=('monetary', 'mean'),
            群組總營收=('monetary', 'sum'))
       .round(1)
       .sort_values('群組總營收', ascending=False)
)
segment_summary['營收佔比_%'] = (
    segment_summary['群組總營收'] / segment_summary['群組總營收'].sum() * 100
).round(1)
segment_summary['客戶佔比_%'] = (
    segment_summary['客戶數'] / segment_summary['客戶數'].sum() * 100
).round(1)
segment_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie(segment_summary['客戶數'], labels=segment_summary.index,
            autopct='%1.1f%%', startangle=90)
axes[0].set_title('客戶數佔比')

axes[1].barh(segment_summary.index, segment_summary['群組總營收'], color='#1565C0')
axes[1].set_title('群組總營收 (R$)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'rfm_segments.png'), dpi=150, bbox_inches='tight')
plt.show()

segment_summary.to_csv(os.path.join(OUTPUT_DIR, 'rfm_segments.csv'))

**Insight**
- Olist 客戶以「一次性購買」為主，**冠軍/忠誠客戶佔比偏低**，平均 F 多落在 1 次
- 「已流失」群組佔客戶數比例最高（R > 中位數），代表平台**回購機制薄弱**
- 「流失風險」群組雖然人數少，但平均 M 較高，是優先挽回的對象

**Recommendation**

| 分群 | 行銷策略 |
|---|---|
| 冠軍客戶 | VIP 制度、新品優先試用、推薦獎勵 |
| 忠誠客戶 | 訂閱制、回購折扣 |
| 潛力新客 | 第二次購買誘因（首購回饋、48 小時限時券） |
| 流失風險 | 個人化召回 EDM、限時免運 |
| 已流失 | 大額折扣喚回 + 滿意度問卷找原因 |
| 一般客戶 | 商品推薦、加購建議提高 AOV |

## 8. 總結

| 主題 | 關鍵發現 | 行動建議 |
|---|---|---|
| 營收 | 2017 Q4 為高峰，2018 進入穩定期 | Q4 前置行銷預算、強化 Top 3 品類 |
| 品類 | 健康美妝、手錶禮品、寢具家飾為三大支柱 | 主推品類加碼、長尾品類淘汰 |
| 評分 | 5 星 ~58%，但 1 星仍有 ~12% | 1 星訂單交叉分析物流延遲與商品描述 |
| 物流 | SP 8.8 天，RN 19.3 天，差距明顯 | 複製 SP 倉配模式、偏遠州 ETA 改為更積極數字 |
| 客戶 | 一次性購買為主，回購薄弱 | 第二次購買誘因 + 流失風險挽回方案 |

## 後續可延伸
- Cohort 分析：每月新客的 N 個月後留存率
- 商品評分與物流天數的相關性檢定
- 月營收時間序列預測（Prophet / ARIMA）